# Write Rajnovic 2019 OD600 as linked ATST

Run from this folder. One study links the ten bacterial-concentration sheets and the LB-only blank.


In [1]:
from pathlib import Path
import datetime
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from ATST import ATSTFile, MultiReadoutATST, validate_atst, write_multi_readout_atst
from ATST.blocks import Assay, Entities, EntityTable, FileMetadata, Layout, Metadata, Readings, ReadoutIds, Study

root=Path.cwd()
workbook=load_workbook(root/"sources/pone.0216292.s002.xlsx",read_only=True,data_only=True)
file_info=FileMetadata(data={"file_name":"atst_rajnovic_2019.atst.txt","format":"ATST","format_version":"0.1","created_on":datetime.date.today().isoformat(),"field_delimiter":"TAB","encoding":"UTF-8"})
study=Study(data={"title":"Fast phage detection and quantification: An optical density-based approach","study_id":"doi:10.1371/journal.pone.0216292","authors":"Denis Rajnovic; Xavier Muñoz-Berbel; Jordi Mas","publication_date":"2019-05-09","publication_doi":"10.1371/journal.pone.0216292","description":"Ten bacteria-by-phage concentration grids and an LB blank from S1 Dataset; OD600 only."})
entities=Entities(tables={name:EntityTable(name="ENTITIES",table_name=name,pk=pk,data=pd.read_csv(root/"entities_definitions"/f"{name}.csv",dtype=str,keep_default_na=False)) for name,pk in [("ISOLATES","ISOLATE_ID"),("PHAGES","PHAGE_ID"),("MEDIA","MEDIUM_ID")]})

def curve_id(index):
    letters=""
    while index:
        index,remainder=divmod(index-1,26)
        letters=chr(65+remainder)+letters
    return "curve_"+letters

def numeric(value):
    if value is None:return None
    if isinstance(value,(int,float)):return value
    if isinstance(value,str) and value.endswith("*"):
        return float(value[:-1].replace(",","."))
    raise ValueError(f"Unexpected OD600 value: {value!r}")

readouts={}
for sheet in list(workbook)[1:]:
    blank=sheet.title=="0 cfu per ml"
    readout_id="LB_blank" if blank else "CFU_"+sheet.title.split(" cfu")[0].replace(".","p").replace("^", "e").replace("x", "x")
    rows=[(i,row) for i,row in enumerate(sheet.iter_rows(min_row=3,values_only=True),3) if isinstance(row[0],(int,float))]
    readings=pd.DataFrame({"Time":[row[0] for _,row in rows]})
    layout_rows=[];starred=[];missing=[]
    groups=[(2,"0 pfu/ml")] if blank else [(2+3*i,sheet.cell(2,2+3*i).value) for i in range(9)]
    for first,label in groups:
        for replicate in range(1,4):
            column=first+replicate-1
            cid=curve_id(len(layout_rows)+1)
            values=[]
            for row_index,row in rows:
                value=row[column-1]
                if isinstance(value,str) and value.endswith("*"):starred.append(f"{get_column_letter(column)}{row_index}")
                if value is None:missing.append(f"{get_column_letter(column)}{row_index}")
                values.append(numeric(value))
            readings[cid]=values
            layout_rows.append({"curve_id":cid,"type":"blank" if blank else ("uninfected_control" if label=="0 pfu/ml" else "phage_exposed"),"isolate_id":"" if blank else "E_coli_DSMZ_613","phage_id":"" if blank or label=="0 pfu/ml" else "T4","bacteria_cfu_ml":"0" if blank else sheet.title.split(" cfu")[0],"pfu_ml":label.split(" pfu")[0],"replicate":str(replicate)})
    assay=Assay(data={"readout_type":"absorbance","readout_unit":"od600","time_unit":"min","medium_id":"LB","culture_type":"planktonic","temperature_c":"37","instrument":"Varioskan Flash plate reader (Thermo Scientific)","plate":"transparent 96-well plate","volumes":"Paper reports 160 µL LB + 20 µL phage solution + 20 µL bacterial solution + 20 µL PB per assay; reported total is 220 µL","source_reference":"doi:10.1371/journal.pone.0216292"})
    metadata=Metadata(data={"source_file":"read_shared_data/sources/pone.0216292.s002.xlsx","source_time_column":"A","source_data_rows":f"3:{rows[-1][0]}","source_condition_columns":"B:D, E:G, H:J, K:M, N:P, Q:S, T:V, W:Y, Z:AB" if not blank else "B:D","source_starred_cells":", ".join(starred),"source_missing_cells":", ".join(missing),"conversion":"Asterisk-suffixed decimal-comma text parsed as numeric OD600; source blanks retained as missing readings"})
    readouts[readout_id]=ATSTFile(file_info=file_info,study=study,metadata=metadata,assay=assay,entities=entities,layout=Layout(data=pd.DataFrame(layout_rows)),readings=Readings(data=readings))
def merged_name(members):
    return "CFU_combined"

# Combine readouts only when metadata, assay, and recorded times match.
groups_by_context = {}
for readout_id, item in readouts.items():
    key = (tuple(item.metadata.data.items()), tuple(item.assay.data.items()), tuple(item.readings.data["Time"].astype(str)))
    groups_by_context.setdefault(key, []).append((readout_id, item))
combined = {}
for members in groups_by_context.values():
    if len(members) == 1:
        combined[members[0][0]] = members[0][1]
        continue
    layouts = []
    readings = members[0][1].readings.data[["Time"]].copy()
    for _, item in members:
        rename = {}
        for old_id in item.layout.data["curve_id"]:
            letter = get_column_letter(len(readings.columns) + len(rename))
            rename[old_id] = "curve_ID_238" if letter == "ID" else "curve_" + letter
        layout = item.layout.data.copy()
        layout["curve_id"] = layout["curve_id"].map(rename)
        layouts.append(layout)
        readings = pd.concat([readings, item.readings.data.drop(columns="Time").rename(columns=rename).reset_index(drop=True)], axis=1)
    first = members[0][1]
    combined[merged_name(members)] = ATSTFile(file_info=file_info, study=study, metadata=first.metadata, assay=first.assay, entities=entities, layout=Layout(data=pd.concat(layouts, ignore_index=True)), readings=Readings(data=readings))
readouts = combined

readout_ids=ReadoutIds(data=pd.DataFrame({"readout_id":list(readouts)}))
for item in readouts.values():item.readout_ids=readout_ids
bundle=MultiReadoutATST(file_info,study,readout_ids,readouts,entities)
validate_atst(bundle)
display(pd.DataFrame([{"readout_id":k,"curves":len(v.layout.data),"time_points":len(v.readings.data)} for k,v in readouts.items()]))


     readout_id  curves  time_points
0  CFU_combined      81           35
1      CFU_10e8      27           51
2    CFU_5x10e7      27           28
3      CFU_10e7      27           28
4    CFU_5x10e6      27           28
5      CFU_10e6      27           19
6    CFU_5x10e5      27           67
7      CFU_10e5      27           65
8      LB_blank       3            3


In [2]:
output=write_multi_readout_atst(bundle,root.parent/"atst_rajnovic_2019",linked_files=True)
print(output)

# Reuse identical metadata payloads across readouts even when assays or times differ.
manifest = output.read_text()
metadata_files = {}
for readout_id, item in bundle.readouts.items():
    signature = tuple(item.metadata.data.items())
    path = f"metadata/meta_{readout_id}.tsv"
    if signature in metadata_files:
        manifest = manifest.replace(f"file={path}", f"file={metadata_files[signature]}")
        (output.parent / path).unlink()
    else:
        metadata_files[signature] = path
output.write_text(manifest)


/Users/juanmantilla/Documents/TKI/ATST/examples/Public data /Rajnovic 2019/atst_rajnovic_2019/atst_rajnovic_2019.atst.txt
